# 1. Collection and data quality

Load every hourly snapshot, check coverage (missing hours, failed games, unresolved Twitch names), and build tidy hourly panels: game × hour for Steam, stream × hour for Twitch.

In [ ]:
import duckdb
import pandas as pd
from pathlib import Path

ROOT = Path.cwd().parent
COLLECTED = ROOT / "data" / "collected"
con = duckdb.connect()

In [ ]:
steam = con.sql(f"SELECT * FROM read_csv_auto('{COLLECTED}/**/steam_*.csv', union_by_name=true)").df()
steam["snapshot_utc"] = pd.to_datetime(steam["snapshot_utc"])

In [ ]:
twitch = con.sql(f"SELECT * FROM read_csv_auto('{COLLECTED}/**/twitch_*.csv', union_by_name=true)").df()
twitch["snapshot_utc"] = pd.to_datetime(twitch["snapshot_utc"])

In [ ]:
# Coverage: snapshots per day, and any rows with errors
steam.groupby(steam.snapshot_utc.dt.date).snapshot_utc.nunique()

In [ ]:
steam[steam['error'].fillna('').ne('')]

In [ ]:
# Hourly new reviews (first difference of the cumulative total)
steam = steam.sort_values(['steam_appid', 'snapshot_utc'])
steam['new_reviews'] = steam.groupby('steam_appid')['total_reviews'].diff()